<a href="https://colab.research.google.com/github/Ashprakash/groundfin/blob/main/benchmark/groundfin_colab_runner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GROUNDFIN Colab Runner

This notebook is meant to stay stable. It pulls the latest code from GitHub, so most fixes should happen in `.py` files instead of requiring notebook replacement.

## 1. Pull Latest Project Code

Rerun this cell whenever the repo changes.

In [ ]:
%cd /content
!test -d groundfin/.git && git -C groundfin pull || git clone https://github.com/Ashprakash/groundfin.git
%cd /content/groundfin
!pip -q install -r requirements-colab.txt
# peft's LoRA dispatcher raises on Colab's old torchao (0.10 < 0.16); it is unused by our
# LoRA setup, so remove it to avoid the ImportError during SFT/GRPO.
!pip -q uninstall -y torchao 2>/dev/null || true
!git log --oneline -1


## 2. Load FinanceBench

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'  # reduce fragmentation OOM

import importlib
from benchmark import financebench_pilot as pilot

importlib.reload(pilot)
df = pilot.load_financebench()
pilot.summarize_dataset(df)
display(df.head(3))

### 2b. Environment & GPU Check

Prints library versions and live VRAM so you can see what the adaptive trainer will pick.
Training auto-selects batch/seq/quantization from free memory and retries smaller on OOM.


In [ ]:
from benchmark import train_groundfin as train
train.print_env()
print()
for m in ['Qwen/Qwen2.5-0.5B-Instruct', 'Qwen/Qwen2.5-3B-Instruct', 'Qwen/Qwen2.5-7B-Instruct']:
    print(m.split('/')[-1], '->', train.recommended_sft_config(m))


## 3. Inspect Prompt

In [ ]:
print(pilot.make_prompt(df.iloc[0], with_evidence=True)[:2500])

## 4. Optional OpenAI Baseline

Set `OPENAI_API_KEY` in Colab secrets or uncomment the environment line below.

In [ ]:
import os

# os.environ['OPENAI_API_KEY'] = 'your_key_here'

if os.environ.get('OPENAI_API_KEY'):
    results, summary = pilot.run_openai_baseline(df, n_examples=20, model='gpt-4.1-mini')
    display(summary)
    display(results.head())
else:
    print('OPENAI_API_KEY is not set. Skipping API baseline.')

## 5. Open-Model Baseline

Use this when you do not have an API key. For best speed, set Colab to `Runtime -> Change runtime type -> T4 GPU`. The default model is small so we can validate the pipeline quickly.


In [ ]:
HF_MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
N_EXAMPLES = 5

hf_results, hf_summary = pilot.run_hf_baseline(
    df,
    n_examples=N_EXAMPLES,
    model_id=HF_MODEL_ID,
    max_new_tokens=160,
    max_evidence_chars=6000,
)

hf_summary_reset = hf_summary.reset_index()
hf_preview = hf_results[['condition', 'gold_answer', 'parsed_answer', 'weak_match_answer', 'numeric_match_answer', 'refusal', 'prediction']].head(10)

print('=== HF SUMMARY ===')
print(hf_summary_reset.to_string(index=False))
print('\n=== HF PREVIEW ===')
print(hf_preview.to_string(index=False))

display(hf_summary)
display(hf_preview)

hf_summary_reset.to_csv('hf_summary.csv', index=False)
hf_results.to_csv('hf_results.csv', index=False)
print('\nWrote hf_summary.csv and hf_results.csv')


## 6. Grounding Probe

This directly tests the GROUNDFIN idea with four conditions: gold evidence, missing evidence, direct grounded evidence, and direct counterfactual evidence. Start tiny; this is a probe, not paper-grade evaluation yet.


In [ ]:
PROBE_MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
N_PROBE_EXAMPLES = 3

probe_results, probe_summary = pilot.run_hf_grounding_probe(
    df,
    n_examples=N_PROBE_EXAMPLES,
    model_id=PROBE_MODEL_ID,
    max_new_tokens=160,
    max_evidence_chars=6000,
)

probe_summary_reset = probe_summary.reset_index()
probe_preview_cols = ['condition', 'target_answer', 'parsed_answer', 'probe_success', 'weak_match_answer', 'numeric_match_answer', 'refusal', 'prediction']
if 'compression_source' in probe_results.columns:
    probe_preview_cols.insert(1, 'compression_source')
probe_preview = probe_results[probe_preview_cols].head(20)

print('Probe conditions:', sorted(probe_results['condition'].unique().tolist()))
print('=== PROBE SUMMARY ===')
print(probe_summary_reset.to_string(index=False))
print('\n=== PROBE PREVIEW ===')
print(probe_preview.to_string(index=False))

display(probe_summary)
display(probe_preview)

probe_summary_reset.to_csv('probe_summary.csv', index=False)
probe_results.to_csv('probe_results.csv', index=False)
print('\nWrote probe_summary.csv and probe_results.csv')


## 7. Template Reliability Comparison

This runs the abstract-aligned method probe: raw evidence, length-matched compact summary, deterministic trace, template without probabilities, full risk-calibrated template, and missing-evidence risk template. It reports accuracy plus reliability metrics such as confidence, Brier score, ECE, and overconfident-wrong rate.


In [ ]:
TEMPLATE_MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
N_TEMPLATE_EXAMPLES = 3

template_results, template_summary = pilot.run_hf_template_comparison(
    df,
    n_examples=N_TEMPLATE_EXAMPLES,
    model_id=TEMPLATE_MODEL_ID,
    max_new_tokens=160,
)

template_summary_reset = template_summary.reset_index()
template_preview = template_results[['condition', 'template_source', 'target_answer', 'parsed_answer', 'template_success', 'weak_match_answer', 'numeric_match_answer', 'refusal', 'confidence', 'brier', 'overconfident_wrong', 'prediction']].head(24)

print('Template conditions:', sorted(template_results['condition'].unique().tolist()))
print('=== TEMPLATE SUMMARY ===')
print(template_summary_reset.to_string(index=False))
print('\n=== TEMPLATE PREVIEW ===')
print(template_preview.to_string(index=False))

display(template_summary)
display(template_preview)

template_summary_reset.to_csv('template_summary.csv', index=False)
template_results.to_csv('template_results.csv', index=False)
print('\nWrote template_summary.csv and template_results.csv')


## 7b. Scaled Template Reliability Suite (paper-grade)

This is the run that decides whether the method is real. It sweeps **multiple models x multiple seeds x n=50+ examples**, in two modes:

- `deployment`: rule/teacher templates built from **raw evidence only, no gold answer** (the main paper claim).
- `oracle`: templates that embed the gold answer (upper bound only; forbidden as a main claim).

A pre-flight leakage check confirms no deployment condition injects the answer (`answer_injected_rate` must be 0.000 for every deployment row). Set Colab to a T4 GPU; this takes a while. Bump `SUITE_N` to 100-200 if the runtime allows.

In [ ]:
# Start with a SMOKE TEST (cheap). Scale up only after it produces a clean table.
SUITE_MODELS = ['Qwen/Qwen2.5-0.5B-Instruct']
SUITE_SEEDS = [7]
SUITE_N = 5            # smoke test: 1 model x 1 seed x 2 modes x 5 ex x 6 cond = 60 gens
SUITE_MODES = ['deployment', 'oracle']
# --- Scale up after the smoke test works ---
# Step 1 (0.5B, real): SUITE_SEEDS = [7, 13, 29]; SUITE_N = 50
# Step 2 (add 1.5B):   SUITE_MODELS = ['Qwen/Qwen2.5-0.5B-Instruct', 'Qwen/Qwen2.5-1.5B-Instruct']
# Full 2x3x2x50x6 ~= 3600 generations, so only run that once the 0.5B tables look right.
# Pre-flight: confirm no deployment condition injects the gold answer.
leak_check = pilot.check_deployment_leakage(df, n_examples=SUITE_N, seed=SUITE_SEEDS[0])
print('Deployment leakage violations (should be 0):', len(leak_check))
display(leak_check)

suite_results = pilot.run_template_comparison_suite(
    df,
    model_ids=SUITE_MODELS,
    seeds=SUITE_SEEDS,
    n_examples=SUITE_N,
    leakage_modes=SUITE_MODES,
    max_new_tokens=160,
    max_evidence_chars=6000,
)

suite_agg, suite_per_seed = pilot.aggregate_template_suite(suite_results)
suite_table = pilot.format_template_suite_table(suite_agg)

print('=== TEMPLATE RELIABILITY SUITE (mean +/- std over seeds) ===')
print(suite_table.to_string(index=False))
display(suite_table)

suite_results.to_csv('suite_results.csv', index=False)
suite_table.to_csv('suite_table.csv', index=False)
suite_per_seed.to_csv('suite_per_seed.csv')
print('\nWrote suite_results.csv, suite_table.csv, suite_per_seed.csv')

### 7c. Selective Accuracy (the headline)

Accuracy when the model answers only its most-confident cases and abstains on the rest.
If the risk-calibrated template's confidence tracks correctness, `acc@30`/`acc@50` should
be far above `acc@100` (full accuracy). This is the honest route to a 75%+ number on a small model.


In [ ]:
dep = suite_results[suite_results['leakage_mode'] == 'deployment']
sel = pilot.selective_summary(dep, ['model_id', 'condition'])
order = {'raw_gold_evidence':0,'length_matched_summary':1,'deterministic_trace':2,
         'template_no_probabilities':3,'risk_calibrated_template':4,'missing_risk_template':5}
sel = sel.assign(_o=sel['condition'].map(order)).sort_values(['model_id','_o']).drop(columns='_o')
print('=== SELECTIVE ACCURACY (deployment) ===')
print(sel.to_string(index=False))
display(sel)
sel.to_csv('selective_suite.csv', index=False)


## 8. Export Pilot Files


In [ ]:
csv_path, jsonl_path = pilot.export_pilot_files(df)
print(csv_path, jsonl_path)

## 8b. Train/Eval Split + Load Training Module

The full method: LoRA SFT under several supervision variants, reliability eval on a
company-disjoint held-out split, then GRPO with a verifiable reward. Needs a T4 GPU.


In [ ]:
import importlib
from benchmark import train_groundfin as train
importlib.reload(train)

train_df, test_df = train.train_test_split(df, test_frac=0.25, seed=7)
print('train rows:', len(train_df), '| test rows:', len(test_df))
print('train companies:', train_df['company'].nunique(), '| test companies:', test_df['company'].nunique())
print('company overlap (must be 0):',
      len(set(train_df['company']) & set(test_df['company'])))

# Sanity-check the method training set: subtasks + no input leakage.
method_ex = train.build_sft_examples(train_df, variant='answer_template_abstain', seed=7)
print('method subtasks:', method_ex['subtask'].value_counts().to_dict())
leak = train.audit_sft_leakage(method_ex)
print('SFT input-leakage rows (must be 0):', len(leak))
display(leak)


## 9. LoRA SFT

Train the two headline conditions first: `answer_only` (baseline) and
`answer_template_abstain` (the method). Add the other variants once these work.
Start small; bump epochs / eval size after a clean run.


In [ ]:
train.free_gpu()  # release VRAM cached by earlier eval/suite cells
SFT_MODEL = 'Qwen/Qwen2.5-0.5B-Instruct'

sft_baseline = train.run_sft(
    train_df, variant='answer_only', model_id=SFT_MODEL,
    epochs=2,
)
sft_method = train.run_sft(
    train_df, variant='answer_template_abstain', model_id=SFT_MODEL,
    epochs=2,
)
print('adapters:', sft_baseline, '|', sft_method)


## 9b. Teacher Distillation (filtered)

Run a strong teacher (7B, 4-bit) over the method prompts, keep only its verifiably-correct
outputs, and SFT the small student on the teacher's own calibrated answers. Reports the
teacher's supported accuracy (the ceiling the student is chasing).


In [ ]:
train.free_gpu()  # release VRAM cached by earlier eval/suite cells
TEACHER_MODEL = 'Qwen/Qwen2.5-7B-Instruct'   # auto 4-bit; drop to 3B if VRAM is tight

teacher_examples = train.generate_teacher_targets(
    train_df, teacher_model_id=TEACHER_MODEL,
)
print('teacher-distill examples:', teacher_examples['subtask'].value_counts().to_dict())

sft_teacher = train.run_sft(
    train_df, variant='teacher_distill', model_id=SFT_MODEL,
    examples=teacher_examples, epochs=2,
)
print('teacher-distilled adapter:', sft_teacher)


## 10. Reliability Evaluation (held-out)

Compare the base model, answer-only SFT, and the method SFT across the
supported / missing / counterfactual sub-tasks. Watch accuracy AND the reliability
metrics (Brier, ECE, overconfident-wrong, abstention refusal_rate).


In [ ]:
EVAL_N = 30   # bump to full test_df once it looks right

specs = [
    {'label': 'base_0.5B',        'model_id': SFT_MODEL, 'adapter': None},
    {'label': 'sft_answer_only',  'model_id': SFT_MODEL, 'adapter': sft_baseline},
    {'label': 'sft_method',       'model_id': SFT_MODEL, 'adapter': sft_method},
]
eval_results, eval_summary = train.compare_models(
    test_df, specs, n_examples=EVAL_N, max_evidence_chars=4000,
)
print('=== RELIABILITY BY MODEL x SUBTASK ===')
print(eval_summary.to_string(index=False))
display(eval_summary)
eval_summary.to_csv('eval_summary.csv', index=False)
eval_results.to_csv('eval_results.csv', index=False)
print('\nWrote eval_summary.csv, eval_results.csv')

# Anchor/teacher-distilled comparisons (uncomment after training them):
#   {'label': 'anchor_3B',    'model_id': 'Qwen/Qwen2.5-3B-Instruct', 'adapter': None},   # 3B anchor (auto 4-bit)
#   {'label': 'sft_teacher',  'model_id': SFT_MODEL, 'adapter': sft_teacher},


### 10b. Selective Accuracy on Held-Out (method vs baselines)

The paper's headline table: accuracy@coverage on the supported subtask for each model.
GROUNDFIN should dominate answer-only at low coverage (answers confidently, abstains otherwise).


In [ ]:
supported = eval_results[eval_results['subtask'] == 'supported']
sel_eval = pilot.selective_summary(supported, ['label'])
print('=== SELECTIVE ACCURACY, held-out supported subtask ===')
print(sel_eval.to_string(index=False))
display(sel_eval)
sel_eval.to_csv('selective_eval.csv', index=False)


## 11. GRPO (verifiable reward)

Continue from the method SFT adapter and optimize the verifiable reward
(answer correctness + calibration + abstention + counterfactual obedience).
This is the heaviest step; keep `num_generations` and the sample small first.


In [ ]:
train.free_gpu()
grpo_dir = train.run_grpo(
    train_df,
    model_id=SFT_MODEL,
    sft_adapter=sft_method,   # start from the method SFT checkpoint
    epochs=1,
    num_generations=4,
    max_evidence_chars=4000,
)

# Re-evaluate the GRPO model against the SFT method and base.
specs_grpo = [
    {'label': 'sft_method',  'model_id': SFT_MODEL, 'adapter': sft_method},
    {'label': 'grpo_method', 'model_id': SFT_MODEL, 'adapter': grpo_dir},
]
grpo_results, grpo_summary = train.compare_models(
    test_df, specs_grpo, n_examples=EVAL_N, max_evidence_chars=4000,
)
print('=== SFT vs GRPO (held-out) ===')
print(grpo_summary.to_string(index=False))
display(grpo_summary)
grpo_summary.to_csv('grpo_summary.csv', index=False)
